# Text Classification with Keras: From Bag of Words to Transformers

This notebook demonstrates various text classification techniques using the Keras API, ranging from simple Bag of Words models to more advanced Transformer-based architectures. We'll be working with the IMDB movie review dataset to classify reviews as positive or negative.

In [1]:
import re
import string
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

def custom_standardization_fn(string_tensor):
  lowercase_string = tf.strings.lower(string_tensor)
  return tf.strings.regex_replace(lowercase_string, f"[{re.escape(string.punctuation)}]", "")

def custom_split_fn(string_tensor):
  return tf.strings.split(string_tensor)

text_vectorization = TextVectorization(
    output_mode="int",
    standardize=custom_standardization_fn,
    split=custom_split_fn,)

## 1. Custom Text Vectorization Layer

We begin by defining a custom `TextVectorization` layer with custom standardization and splitting functions. This allows us to preprocess text by converting it to lowercase and removing punctuation, and then splitting it into individual words. This layer will be used to convert raw text into numerical representations that can be fed into our models.

In [ ]:
dataset = [
"I write, erase, rewrite",
"Erase again, and then",
"A poppy blooms.",
]

text_vectorization.adapt(dataset)

After defining the `TextVectorization` layer, we adapt it to a small sample dataset. This step builds the vocabulary of the layer, mapping each unique word to an integer index. The `get_vocabulary()` method then allows us to inspect the learned vocabulary.

In [ ]:
text_vectorization.get_vocabulary()

['',
 '[UNK]',
 np.str_('erase'),
 np.str_('write'),
 np.str_('then'),
 np.str_('rewrite'),
 np.str_('poppy'),
 np.str_('i'),
 np.str_('blooms'),
 np.str_('and'),
 np.str_('again'),
 np.str_('a')]

In [ ]:
vocabulary = text_vectorization.get_vocabulary()
test_sentence = "I write, rewrite and STILL REWRITE AGAIN"
encoded_sentence = text_vectorization(test_sentence)
print(encoded_sentence)

tf.Tensor([ 7  3  5  9  1  5 10], shape=(7,), dtype=int64)


Here, we demonstrate how the `TextVectorization` layer encodes a new sentence using the vocabulary it learned. We also show how to decode the numerical representation back into text using an inverse vocabulary mapping, highlighting how unknown words (`[UNK]`) are handled.

In [ ]:
inverse_vocab = dict(enumerate(vocabulary))
decoded_sentence = " ".join(inverse_vocab[int(i)] for i in encoded_sentence)
print(decoded_sentence)

i write rewrite and [UNK] rewrite again


### preparing the imdb moview reviews dataset

## 2. Preparing the IMDB Movie Reviews Dataset

Next, we download and preprocess the IMDB movie reviews dataset. This dataset consists of 50,000 movie reviews, labeled as either positive or negative. We'll use 20,000 reviews for training, 5,000 for validation, and 25,000 for testing.

In [2]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  63.8M      0  0:00:01  0:00:01 --:--:-- 63.9M


We download the dataset, extract it, and remove the `unsup` (unsupervised) directory as it's not needed for our supervised classification task.

In [3]:
!rm -r aclImdb/train/unsup

In [4]:
!cat aclImdb/train/pos/4077_10.txt

I first saw this back in the early 90s on UK TV, i did like it then but i missed the chance to tape it, many years passed but the film always stuck with me and i lost hope of seeing it TV again, the main thing that stuck with me was the end, the hole castle part really touched me, its easy to watch, has a great story, great music, the list goes on and on, its OK me saying how good it is but everyone will take there own best bits away with them once they have seen it, yes the animation is top notch and beautiful to watch, it does show its age in a very few parts but that has now become part of it beauty, i am so glad it has came out on DVD as it is one of my top 10 films of all time. Buy it or rent it just see it, best viewing is at night alone with drink and food in reach so you don't have to stop the film.<br /><br />Enjoy

This cell shows an example of a positive movie review from the dataset, demonstrating the raw text format we will be working with.

In [5]:
#creating a validation data set by setting apart 20% of training text
import os, pathlib, shutil, random
base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"

for category in ("neg", "pos"):
  os.makedirs(val_dir / category)
  files = os.listdir(train_dir / category)
  random.Random(1337).shuffle(files)
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname,
                val_dir / category / fname)

To ensure robust model evaluation, we split the training data further into training and validation sets. Specifically, we set aside 20% of the original training data for validation, moving these files to a new 'val' directory.

In [6]:
from tensorflow import keras
batch_size = 32

train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


Using Keras' `text_dataset_from_directory` utility, we load the IMDB dataset into `tf.data.Dataset` objects for efficient processing. This creates `train_ds`, `val_ds`, and `test_ds` for our models.

In [8]:
for inputs, targets in train_ds:
  print("inputs.shape:", inputs.shape)
  print("inputs.dtype:", inputs.dtype)
  print("targets.shape:", targets.shape)
  print("targets.dtype:", targets.dtype)
  print("inputs[0]:", inputs[0])
  print("targets[0]:", targets[0])
  break

inputs.shape: (32,)
inputs.dtype: <dtype: 'string'>
targets.shape: (32,)
targets.dtype: <dtype: 'int32'>
inputs[0]: tf.Tensor(b"I am so disappointed. This movie left me feeling jipped out of my time and mental energy. Here was the quintessential Woody Allen film all over again: the neurotic upper-class Manhattanites debating whether or not they will cheat on their spouses. Woody, I've seen these characters already, I've seen the storyline from you ten times already. Where did your creativity go??? You need to open your eyes and look around you. The world has changed dramatically since Annie Hall - and you need to change along with it.<br /><br />There are far more interesting and funny scenarios to which you can apply your brand of angst and neuroticism - why not try them out instead of rehashing the same old slop over and over and over again.<br /><br />When I hear that Woody Allen has a new project coming out, it does nothing for me - because now I've come to expect his old standby: 

This cell inspects the format of the loaded dataset, showing the shapes, data types, and an example of a single input review (text) and its corresponding target label (0 for negative, 1 for positive).

## 3. Bag of Words Models using N-grams

In this section, we implement various Bag of Words (BoW) models. BoW models represent text as a collection of word occurrences within a document, disregarding grammar and word order. We will explore unigrams (single words), bigrams (pairs of words), and TF-IDF (Term Frequency-Inverse Document Frequency) weighting.

In [9]:
#preprocessing our dataset with TextVectorization layer

text_vectorization = TextVectorization(
    max_tokens=20000,
    output_mode="multi_hot")

text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)

binary_1gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

binary_1gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

binary_1gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

Here, we configure `TextVectorization` to output 'multi_hot' encoded vectors for 1-grams (individual words) and adapt it to our training data. The `multi_hot` encoding creates a binary vector where each dimension corresponds to a word in the vocabulary, indicating its presence (1) or absence (0) in the document. We then apply this vectorization to our training, validation, and test datasets.

In [10]:
for inputs, targets in binary_1gram_train_ds:
  print("inputs.shape:", inputs.shape)
  print("inputs.dtype:", inputs.dtype)
  print("targets.shape:", targets.shape)
  print("targets.dtype:", targets.dtype)
  print("inputs[0]:", inputs[0])
  print("targets[0]:", targets[0])
  break

inputs.shape: (32, 20000)
inputs.dtype: <dtype: 'int64'>
targets.shape: (32,)
targets.dtype: <dtype: 'int32'>
inputs[0]: tf.Tensor([1 1 1 ... 0 0 0], shape=(20000,), dtype=int64)
targets[0]: tf.Tensor(1, shape=(), dtype=int32)


This cell provides a peek into the structure of the `binary_1gram_train_ds`. We can see the input shape (batch_size, max_tokens) and the `multi_hot` encoded vector for a single review.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def get_model(max_tokens=20000, hidden_dim=16):
  inputs = keras.Input(shape=(max_tokens,))
  x = layers.Dense(hidden_dim, activation="relu")(inputs)
  x = layers.Dropout(0.5)(x)
  outputs = layers.Dense(1, activation="sigmoid")(x)
  model = keras.Model(inputs, outputs)
  model.compile(optimizer="rmsprop",
                loss="binary_crossentropy",
                metrics=["accuracy"])
  return model

We define a simple neural network model architecture (`get_model`) to be used with our Bag of Words representations. This model consists of a `Dense` layer with ReLU activation, followed by a `Dropout` layer for regularization, and a final `Dense` layer with a sigmoid activation for binary classification.

In [ ]:
model = get_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

We instantiate and display the summary of the 1-gram model, showing its layers and parameter count.

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("binary_1gram.keras",
                                    save_best_only=True)
    ]

model.fit(binary_1gram_train_ds.cache(),
          validation_data=binary_1gram_val_ds.cache(),
          epochs=10,
          callbacks=callbacks)

model = keras.models.load_model("binary_1gram.keras")
print(f"Test acc: {model.evaluate(binary_1gram_test_ds)[1]:.3f}")

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 16s 24ms/step - accuracy: 0.8284 - loss: 0.4166 - val_accuracy: 0.8886 - val_loss: 0.2881
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.8967 - loss: 0.2801 - val_accuracy: 0.8924 - val_loss: 0.2799
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.9150 - loss: 0.2462 - val_accuracy: 0.8896 - val_loss: 0.2930
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9204 - loss: 0.2325 - val_accuracy: 0.8860 - val_loss: 0.3080
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.9288 - loss: 0.2174 - val_accuracy: 0.8854 - val_loss: 0.3228
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9305 - loss: 0.2072 - val_accuracy: 0.8850 - val_loss: 0.3421
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9365 - loss: 0.1982 - val_accuracy: 0.8822 - val_loss: 0.3488
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9361 - loss: 0.1994 - val_a

The model is trained using the `binary_1gram_train_ds` and validated on `binary_1gram_val_ds`. We use `ModelCheckpoint` to save the best model based on validation accuracy. Finally, the best model is loaded and evaluated on the test dataset.

In [ ]:
#configuring the TextVectorization to return bigrams

text_vectorization = TextVectorization(
ngrams=2,
max_tokens=20000,
output_mode="multi_hot",
)

text_vectorization.adapt(text_only_train_ds)

binary_2gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

binary_2gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

binary_2gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

model = get_model()
model.summary()

callbacks = [
keras.callbacks.ModelCheckpoint("binary_2gram.keras",
save_best_only=True)]

model.fit(binary_2gram_train_ds.cache(),
          validation_data=binary_2gram_val_ds.cache(),
          epochs=10,
          callbacks=callbacks)

model = keras.models.load_model("binary_2gram.keras")
print(f"Test acc: {model.evaluate(binary_2gram_test_ds)[1]:.3f}")

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.8457 - loss: 0.3764 - val_accuracy: 0.8966 - val_loss: 0.2667
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - accuracy: 0.9186 - loss: 0.2334 - val_accuracy: 0.9002 - val_loss: 0.2690
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.9362 - loss: 0.2011 - val_accuracy: 0.8994 - val_loss: 0.2867
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9456 - loss: 0.1784 - val_accuracy: 0.8944 - val_loss: 0.3175
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.9487 - loss: 0.1708 - val_accuracy: 0.8946 - val_loss: 0.3203
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9531 - loss: 0.1601 - val_accuracy: 0.8930 - val_loss: 0.3366
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9557 - loss: 0.1572 - val_accuracy: 0.8880 - val_loss: 0.3516
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.9575 - loss: 0.1554 - val_

In this section, we extend our Bag of Words approach to use **bigrams** (sequences of two words). The `TextVectorization` layer is reconfigured with `ngrams=2`, allowing it to capture some local word order information. We then repeat the process of adapting the layer, creating the `binary_2gram` datasets, defining a new model, and training and evaluating it.

In [ ]:
#configuring Textvectorization to return TF-IDF weights

text_vectorization = TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode="tf_idf",)

text_vectorization.adapt(text_only_train_ds)
tfidf_2gram_train_ds = train_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)
tfidf_2gram_val_ds = val_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)
tfidf_2gram_test_ds = test_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)

Here, we switch to a **TF-IDF** (Term Frequency-Inverse Document Frequency) weighting scheme for our bigram representation. Instead of just binary presence, `output_mode="tf_idf"` assigns weights to words based on how frequently they appear in a document relative to their frequency across all documents. This helps to give more importance to rare but significant terms. The model is then trained and evaluated with this new representation.

In [ ]:
model = get_model()
model.summary()
callbacks = [
keras.callbacks.ModelCheckpoint("tfidf_2gram.keras",
save_best_only=True)
]
model.fit(tfidf_2gram_train_ds.cache(),
validation_data=tfidf_2gram_val_ds.cache(),
epochs=10,
callbacks=callbacks)
model = keras.models.load_model("tfidf_2gram.keras")
print(f"Test acc: {model.evaluate(tfidf_2gram_test_ds)[1]:.3f}")

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.7817 - loss: 0.4962 - val_accuracy: 0.8862 - val_loss: 0.3111
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - accuracy: 0.8612 - loss: 0.3417 - val_accuracy: 0.8908 - val_loss: 0.2879
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.8738 - loss: 0.3050 - val_accuracy: 0.8664 - val_loss: 0.3292
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.8798 - loss: 0.2838 - val_accuracy: 0.8806 - val_loss: 0.3074
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.8852 - loss: 0.2720 - val_accuracy: 0.8722 - val_loss: 0.3187
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.8906 - loss: 0.2599 - val_accuracy: 0.8858 - val_loss: 0.3134
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.8974 - loss: 0.2474 - val_accuracy: 0.8856 - val_loss: 0.3350
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9011 - loss: 0.2431 - val_accura

The model with TF-IDF weighted bigrams is instantiated, its summary displayed, and then trained and evaluated. This allows us to compare the performance of TF-IDF against simple multi-hot encoding.

### Sequence Models

## 4. Sequence Models

Moving beyond Bag of Words, we now explore sequence models that can capture word order and contextual information. These models typically use embeddings to represent words as dense vectors.

In [11]:
#1 training our own embeddings
from tensorflow.keras import layers

max_length = 600
max_tokens = 20000
text_vectorization = layers.TextVectorization(
max_tokens=max_tokens,
output_mode="int",
output_sequence_length=max_length,
)

text_vectorization.adapt(text_only_train_ds)

int_train_ds = train_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)

int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)


### 4.1 Training Our Own Embeddings

First, we configure `TextVectorization` to output integer sequences, where each integer represents a word's index in the vocabulary. We also set a `output_sequence_length` to pad or truncate sequences to a fixed length. This preprocessed data will be fed into an embedding layer that learns word representations during training.

In [ ]:
#a bidirectional RNN with embedded layer with enabled masking

inputs = keras.Input(shape=(None,), dtype="int64")
embedded = layers.Embedding(input_dim=max_tokens, output_dim=256, mask_zero=True)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
loss="binary_crossentropy",
metrics=["accuracy"])

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │  5,120,000 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer_3[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 64)        │     73,984 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 1)         │         65 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,194,049 (19.81 MB)

 Trainable params: 5,194,049 (19.81 MB)

 Non-trainable params: 0 (0.00 B)

We construct a bidirectional RNN (Recurrent Neural Network) model using LSTM (Long Short-Term Memory) layers. This model includes an `Embedding` layer that will learn dense vector representations for each word. The `mask_zero=True` argument in the embedding layer enables masking, allowing the model to ignore padded zeros in the input sequences. A `Bidirectional` wrapper processes the sequence in both forward and backward directions, capturing more context. `Dropout` is used for regularization.

In [ ]:
callbacks = [
keras.callbacks.ModelCheckpoint("embeddings_bidir_gru_with_masking.keras",
save_best_only=True)
]

model.fit(int_train_ds, validation_data=int_val_ds, epochs=10, callbacks=callbacks)

model = keras.models.load_model("embeddings_bidir_gru_with_masking.keras")
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 825s 1s/step - accuracy: 0.7695 - loss: 0.4699 - val_accuracy: 0.8516 - val_loss: 0.3464
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 926s 1s/step - accuracy: 0.8765 - loss: 0.3059 - val_accuracy: 0.8642 - val_loss: 0.3272
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 879s 1s/step - accuracy: 0.9077 - loss: 0.2389 - val_accuracy: 0.8826 - val_loss: 0.3031
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 793s 1s/step - accuracy: 0.9299 - loss: 0.1875 - val_accuracy: 0.8616 - val_loss: 0.3700
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 790s 1s/step - accuracy: 0.9467 - loss: 0.1466 - val_accuracy: 0.8758 - val_loss: 0.3398
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 811s 1s/step - accuracy: 0.9631 - loss: 0.1086 - val_accuracy: 0.8902 - val_loss: 0.3571
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 798s 1s/step - accuracy: 0.9711 - loss: 0.0850 - val_accuracy: 0.8458 - val_loss: 0.5908
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 793s 1s/step - accuracy: 0.9816 - loss: 0.0583 - val_accu

The bidirectional LSTM model with learned embeddings is then trained and evaluated. We use `ModelCheckpoint` to save the best performing model.

In [ ]:
#Using pre trained embeddings(GloVe)

!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

import numpy as np
path_to_glove_file = "glove.6B.100d.txt"

embeddings_index = {}
with open(path_to_glove_file) as f:
  for line in f:
    word, coefs = line.split(maxsplit=1)
    coefs = np.fromstring(coefs, "f", sep=" ")
    embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} word vectors.")

--2026-08-20 09:35:13--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-08-20 09:35:13--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-08-20 09:35:13--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

### 4.2 Using Pre-trained Embeddings (GloVe)

Instead of learning embeddings from scratch, we can leverage pre-trained word embeddings like GloVe (Global Vectors for Word Representation). These embeddings are learned from massive text corpora and capture rich semantic relationships between words.

Here, we download the GloVe embeddings, parse the file, and create an `embeddings_index` dictionary mapping words to their corresponding 100-dimensional GloVe vectors.

In [ ]:
embedding_dim = 100
vocabulary = text_vectorization.get_vocabulary()
word_index = dict(zip(vocabulary, range(len(vocabulary))))

embedding_matrix = np.zeros((max_tokens, embedding_dim))
for word, i in word_index.items():
  if i < max_tokens:
    embedding_vector = embeddings_index.get(word)
  if embedding_vector is not None:
    embedding_matrix[i] = embedding_vector

We create an embedding matrix where each row corresponds to a word in our dataset's vocabulary, and the columns contain its pre-trained GloVe vector. Words not found in the GloVe vocabulary will have a zero vector. This matrix will initialize our `Embedding` layer.

In [ ]:
embedding_layer = layers.Embedding(
    max_tokens,
    embedding_dim,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable = False,
    mask_zero = True)

inputs = keras.Input(shape=(None,), dtype="int64")
embedded = embedding_layer(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint("glove_embeddings_sequence_model.keras",
                                    save_best_only=True)]

model.fit(int_train_ds, validation_data=int_val_ds, epochs=10,callbacks=callbacks)
model = keras.models.load_model("glove_embeddings_sequence_model.keras")

print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 100) │  2,000,000 │ input_layer_4[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, None)      │          0 │ input_layer_4[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 64)        │     34,048 │ embedding_1[0][0… │
│ (Bidirectional)     │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 64)        │          0 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 1)         │         65 │ dropout_4[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,034,113 (7.76 MB)

 Trainable params: 34,113 (133.25 KB)

 Non-trainable params: 2,000,000 (7.63 MB)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 500s 789ms/step - accuracy: 0.6801 - loss: 0.5871 - val_accuracy: 0.6768 - val_loss: 0.6258
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 478s 765ms/step - accuracy: 0.7858 - loss: 0.4604 - val_accuracy: 0.8210 - val_loss: 0.3981
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 494s 791ms/step - accuracy: 0.8224 - loss: 0.4026 - val_accuracy: 0.8368 - val_loss: 0.3727
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 503s 792ms/step - accuracy: 0.8411 - loss: 0.3693 - val_accuracy: 0.8472 - val_loss: 0.3519
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 497s 786ms/step - accuracy: 0.8555 - loss: 0.3425 - val_accuracy: 0.8518 - val_loss: 0.3412
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 500s 783ms/step - accuracy: 0.8658 - loss: 0.3217 - val_accuracy: 0.8600 - val_loss: 0.3270
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 491s 785ms/step - accuracy: 0.8762 - loss: 0.3030 - val_accuracy: 0.8648 - val_loss: 0.3222
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 475s 760ms/step - accuracy: 0.8808 -

We define a similar bidirectional LSTM model, but this time, the `Embedding` layer is initialized with our `embedding_matrix` from GloVe. Crucially, `trainable=False` is set to keep these pre-trained embeddings fixed during training, treating them as a feature extraction step. The model is then compiled, trained, and evaluated on the IMDB test set.

### Transformer encoder implemented as a subclassed layer

## 5. Transformer Models

Transformers are state-of-the-art architectures for sequence modeling, largely replacing RNNs in many tasks due to their ability to process sequences in parallel using self-attention mechanisms.

In [12]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class TransformerEncoder(layers.Layer):
  def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
    super().__init__(**kwargs)
    self.embed_dim = embed_dim
    self.dense_dim = dense_dim
    self.num_heads = num_heads
    self.attention = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_dim)
    self.dense_proj = keras.Sequential(
        [layers.Dense(dense_dim, activation="relu"),
         layers.Dense(embed_dim),]
    )
    self.layernorm_1 = layers.LayerNormalization()
    self.layernorm_2 = layers.LayerNormalization()

  def call(self, inputs, mask=None):
    if mask is not None:
      mask = mask[:, tf.newaxis, :]
    attention_output = self.attention(
        inputs, inputs, attention_mask=mask)
    proj_input = self.layernorm_1(inputs + attention_output)
    proj_output = self.dense_proj(proj_input)
    return self.layernorm_2(proj_input + proj_output)

  def get_config(self):
    config = super().get_config()
    config.update({
        "embed_dim": self.embed_dim,
        "num_heads": self.num_heads,
        "dense_dim": self.dense_dim,
    })
    return config

### 5.1 Transformer Encoder Implemented as a Subclassed Layer

We define a `TransformerEncoder` as a custom Keras `Layer`. This encoder consists of:
- A `MultiHeadAttention` layer to allow the model to jointly attend to information from different representation subspaces at different positions.
- Two `LayerNormalization` layers for stabilizing the network.
- A `Sequential` block with `Dense` layers for position-wise feed-forward networks.

The `call` method defines the forward pass, and `get_config` is implemented for serialization.

###Implementing positional Embedding as a subclassed layer

### 5.2 Implementing Positional Embedding as a Subclassed Layer

Transformers process input sequences without inherent order, so we need to inject positional information. We implement a `PositionalEmbedding` layer that combines standard token embeddings with learned positional embeddings. This layer enables the model to understand the relative or absolute position of tokens in the sequence.

The `compute_mask` method ensures that padded tokens are ignored during attention calculations.

In [13]:
class PositionalEmbedding(layers.Layer):
  def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
    super().__init__(**kwargs)
    self.token_embeddings = layers.Embedding(
        input_dim=input_dim,
        output_dim=output_dim)
    self.positional_embeddings = layers.Embedding(
        input_dim=sequence_length,
        output_dim=output_dim)
    self.sequence_length = sequence_length
    self.input_dim = input_dim
    self.output_dim = output_dim

  def call(self, inputs):
    length = tf.shape(inputs)[-1]
    positions = tf.range(start=0, limit=length, delta=1)
    embedded_tokens = self.token_embeddings(inputs)
    embedded_positions = self.positional_embeddings(positions)
    return embedded_tokens + embedded_positions

  def compute_mask(self, inputs, mask=None):
    return tf.keras.ops.not_equal(inputs, 0)
    #return tf.math.not_equal(inputs, 0)

  def get_config(self):
    config = super().get_config()
    config.update({
        "output_dim": self.output_dim,
        "sequence_length": self.sequence_length,
        "input_dim": self.input_dim,
    })
    return config

### 5.3 Assembling and Training the Transformer Model

Finally, we put all the pieces together to construct a full Transformer-based text classification model. The model consists of:
- The `PositionalEmbedding` layer to encode input sequences with positional information.
- The `TransformerEncoder` layer to process the embedded sequences using self-attention.
- A `GlobalMaxPooling1D` layer to downsample the sequence output into a single vector.
- A `Dropout` layer for regularization.
- A final `Dense` layer with sigmoid activation for binary classification.

This model is then compiled, trained for 20 epochs, and evaluated on the test set. Custom objects are provided during model loading to ensure the custom `TransformerEncoder` and `PositionalEmbedding` layers are correctly recognized.

In [14]:
##putting it all together

vocab_size = 20000
sequence_length = 600
embed_dim = 256
num_heads = 2
dense_dim = 32

inputs = keras.Input(shape=(None,), dtype="int64")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(inputs)
x = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint("full_transformer_encoder.keras",
                                    save_best_only=True)
]

model.fit(int_train_ds, validation_data=int_val_ds, epochs=20,
          callbacks=callbacks)

model = keras.models.load_model(
    "full_transformer_encoder.keras",
    custom_objects={"TransformerEncoder": TransformerEncoder,
                    "PositionalEmbedding": PositionalEmbedding})

print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'transformer_encoder' (of type TransformerEncoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 256) │  5,273,600 │ input_layer[0][0] │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encoder │ (None, None, 256) │    543,776 │ positional_embed… │
│ (TransformerEncode… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 256)       │          0 │ transformer_enco… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │        257 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,817,633 (22.19 MB)

 Trainable params: 5,817,633 (22.19 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 57s 74ms/step - accuracy: 0.7459 - loss: 0.5375 - val_accuracy: 0.8598 - val_loss: 0.3341
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 51s 81ms/step - accuracy: 0.8573 - loss: 0.3304 - val_accuracy: 0.8742 - val_loss: 0.2917
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 52s 83ms/step - accuracy: 0.8906 - loss: 0.2669 - val_accuracy: 0.8818 - val_loss: 0.2962
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 52s 83ms/step - accuracy: 0.9109 - loss: 0.2199 - val_accuracy: 0.8620 - val_loss: 0.3363
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 52s 82ms/step - accuracy: 0.9269 - loss: 0.1889 - val_accuracy: 0.8796 - val_loss: 0.3149
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 49s 79ms/step - accuracy: 0.9429 - loss: 0.1521 - val_accuracy: 0.8826 - val_loss: 0.3267
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 52s 82ms/step - accuracy: 0.9574 - loss: 0.1193 - val_accuracy: 0.8832 - val_loss: 0.3923
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 51s 82ms/step - accuracy: 0.9677 - loss: 0.0929 - 

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'positional_embedding', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'transformer_encoder', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'transformer_encoder' (of type TransformerEncoder

782/782 ━━━━━━━━━━━━━━━━━━━━ 17s 20ms/step - accuracy: 0.8739 - loss: 0.2961
Test acc: 0.874


## Summary: Model Performance Comparison

In this notebook, we explored several text classification models for the IMDB movie review dataset, ranging from Bag of Words (BoW) approaches to more complex sequence models like LSTMs with embeddings and Transformers.

While advanced models like LSTMs and Transformers are often expected to outperform simpler BoW models, our experiments on this specific dataset show an interesting outcome:

*   **Bag of Words (1-gram):** Achieved a respectable test accuracy of **0.888**.
*   **Bag of Words (2-gram Multi-Hot):** Achieved a test accuracy of **0.899**.
*   **Bag of Words (2-gram TF-IDF):** Achieved a test accuracy of **0.887**.
*   **Sequence Model (Learned Embeddings Bi-LSTM):** Achieved a test accuracy of **0.870**.
*   **Sequence Model (Pre-trained GloVe Embeddings Bi-LSTM):** Achieved a test accuracy of **0.874**.
*   **Transformer Model:** Achieved a test accuracy of **0.874**.

For *our specific problem* and dataset, the **Bag of Words (bigram) models**, particularly when using TF-IDF weighting, proved to be highly effective. They achieved performance comparable to, and in some cases even slightly surpassing, the more complex sequence models. This suggests that for this IMDB sentiment classification task, the presence and frequency of word pairs (bigrams) provided sufficient signal for accurate classification, and the additional complexity of capturing long-range dependencies or positional information offered by LSTMs and Transformers did not translate into significant performance gains.

This highlights that simpler models can sometimes be just as effective, or even more so, depending on the characteristics of the dataset and the problem at hand, especially when their representations adequately capture the necessary features.